In [ ]:
# =============================================
# NOTEBOOK SETUP — FINAL WORKING VERSION
# =============================================
import sys
from pathlib import Path

# Reliable project root (works from inside notebooks/)
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import duckdb
from config.settings import DB_PATH

print(f"✅ Project root: {PROJECT_ROOT}")
print(f"✅ DB path: {DB_PATH}")

# Connect writable
con = duckdb.connect(str(DB_PATH))
print("✅ Connected in writable mode!")

# Create view using FULL ABSOLUTE PATH (this is what was breaking)
con.sql(f"""
    CREATE OR REPLACE VIEW silver_lineups AS 
    SELECT * 
    FROM read_parquet('{PROJECT_ROOT}/data/silver/lineups/year=*/month=*/day=*/lineups.parquet', 
                      hive_partitioning=1)
""")

print("✅ silver_lineups view created with correct absolute path!")

# Test the count
total_df = con.sql("SELECT COUNT(*) as total FROM silver_lineups").df()
total = total_df.iloc[0]['total']
print(f"✅ Total games in silver_lineups: {total:,}")

# Switch to read-only for the rest of the notebook
con.close()
con = duckdb.connect(str(DB_PATH), read_only=True)
print("✅ Switched to read-only mode — you're good to go!")

In [ ]:
# Most recent lineups with the new bbref columns
con.sql("""
    SELECT 
        game_date,
        away_team,
        home_team,
        away_starter_name,
        away_starter_primary_bbref_id,
        away_lineup_bbref_ids[0:3] AS first_three_away_bbref_ids,
        is_confirmed
    FROM silver_lineups 
    ORDER BY fetch_timestamp DESC 
    LIMIT 5
""").df()

In [ ]:
# Quick check that matching worked
con.sql("""
    SELECT 
        game_date,
        away_team,
        away_starter_name,
        away_starter_primary_bbref_id
    FROM silver_lineups 
    WHERE away_starter_primary_bbref_id IS NOT NULL
    LIMIT 8
""").df()


In [ ]:
# Quick validation — how many players matched?
con.sql("""
    SELECT 
        COUNT(CASE WHEN away_starter_primary_bbref_id IS NOT NULL THEN 1 END) as starters_matched,
        COUNT(*) as total_starters
    FROM silver_lineups
""").df()

In [3]:
# CELL 1 — Setup (exactly like your ducked_test.ipynb)
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import duckdb
from config.settings import DB_PATH

print(f"✅ Project root: {PROJECT_ROOT}")
print(f"✅ Central DB: {DB_PATH}")

con = duckdb.connect(str(DB_PATH))
print("✅ Connected to central time-series DB!")

# CELL 2 — Quick overview
print("\nTables in the DB:")
print(con.sql("SELECT name FROM sqlite_master WHERE type='table'").df())

# CELL 3 — Sample data (most recent first)
con.sql("""
    SELECT fetch_timestamp, sport_key, home_team, away_team, commence_time
    FROM raw_odds 
    ORDER BY fetch_timestamp DESC 
    LIMIT 5
""").df()

con.sql("""
    SELECT fetch_timestamp, game_date, away_team, home_team, away_starter_primary_bbref_id, is_confirmed
    FROM raw_lineups 
    ORDER BY fetch_timestamp DESC 
    LIMIT 5
""").df()

# CELL 4 — Row counts
con.sql("SELECT 'raw_odds' as table, COUNT(*) as rows FROM raw_odds UNION ALL SELECT 'raw_lineups', COUNT(*) FROM raw_lineups").df()

✅ Project root: /Users/patrickmaloney/Documents/mlb-betting
✅ Central DB: /Users/patrickmaloney/Documents/mlb-betting/data/db/mlb_betting.duckdb
✅ Connected to central time-series DB!

Tables in the DB:
          name
0  raw_lineups
1     raw_odds


,table,rows
0,raw_odds,1
1,raw_lineups,17
